# Workshop — Ingestion

**Scenario:** You are a Data Engineer at RetailHub. Load raw customer and order data from files into the Lakehouse, explore it, clean it, and write a Bronze Delta table — following the Medallion Architecture pattern.

## Tasks at a Glance

| # | Topic | Key API | |
|---|-------|---------|---|
| 01 | Setup & imports | `pyspark.sql.types` · `pyspark.sql.functions` |  pre-filled |
| 02 | Load CSV — auto schema | `spark.read.option("inferSchema","true").csv(...)` |  pre-filled |
| 03 | Load CSV — explicit schema | `StructType` · `StructField` | |
| 04 | Explore data | `printSchema` · `describe` · `display` | |
| 05 | Load JSON with schema | `spark.read.format("json").schema(...)` | |
| 06 | Select & filter | `filter(col(...))` · `year(col(...))` | |
| 07 | Add and transform columns | `withColumn` · `cast` · `concat_ws` | |
| 08 | Conditional logic | `when(cond, val).otherwise(val)` | |
| 09 | Aggregations | `groupBy().agg(count, sum, avg)` | |
| 10 | SQL — temporary views | `createOrReplaceTempView` · `spark.sql` | |
| 11 | Write Bronze Delta table | `write.format("delta").saveAsTable` | |
| 12 | ⭐ Bonus — Join | `df.join(other, on, "left")` | |

>  **Tasks 01 and 02 are pre-filled** — run them, then start coding from **Task 03**.

## Learning Objectives

By the end of this workshop you will be able to:

-  Load **CSV** and **JSON** files with inferred and explicit schemas
-  Explore a DataFrame with `printSchema`, `describe`, and `display`
-  Apply PySpark **transformations**: `select`, `filter`, `withColumn`, `cast`, `when/otherwise`
-  Aggregate data with `groupBy`, `count`, `sum`, `avg`
-  Create **temp views** and query them with `spark.sql`
-  Write a **Bronze Delta table** following Medallion Architecture principles
-  Validate each step against automated assertions

## Rules

- Replace `# YOUR CODE HERE` with your solution
- Run the **CHECK** cell after each task — all assertions must pass before moving on
- Use only the Databricks API (`spark`, `display`, `dbutils`) — no external libraries
- Bronze rule (Task 11): **no business logic, no filtering — raw data + metadata only**


In [0]:
%run ../setup/00_setup

In [0]:
CUSTOMERS_CSV  = f"{DATASET_PATH}/customers/customers.csv"
ORDERS_JSON    = f"{DATASET_PATH}/orders/orders_batch.json"
TARGET_TABLE   = f"{CATALOG}.{BRONZE_SCHEMA}.lab_ingestion_customers"

print(f"Customers : {CUSTOMERS_CSV}")
print(f"Orders    : {ORDERS_JSON}")
print(f"Target    : {TARGET_TABLE}")

---
## Task 01 — Imports  Pre-filled

> **This cell is pre-filled — just run it.** It imports all PySpark types and functions used throughout the workshop.

Import the following:
- `StructType`, `StructField` from `pyspark.sql.types`
- `StringType`, `IntegerType`, `DoubleType`, `TimestampType`, `DateType` from `pyspark.sql.types`
- `col`, `lit`, `current_timestamp`, `input_file_name`, `when`, `year`, `count`, `sum`, `avg`, `desc` from `pyspark.sql.functions`

** Guidance — Task 01**

The goal is to make the core PySpark building blocks available in this notebook's namespace.

**Types vs. Functions**
PySpark imports are organised into two main packages. `pyspark.sql.types` holds schema-related classes — you use these when declaring a DataFrame structure explicitly (`StructType`, `StructField`, and individual column types such as `StringType`, `IntegerType`, etc.). `pyspark.sql.functions` holds transformation and aggregation utilities that return `Column` expressions — things like `col()`, `when()`, `year()`, `count()`, `sum()`, `avg()`, and `desc()`.

**Import style**
Use explicit named imports rather than wildcard `from ... import *`. This avoids shadowing Python built-ins — for example, `sum` from `pyspark.sql.functions` is **not** the same as Python's built-in `sum()`. Group the type imports together and the function imports together for readability.

**Things to think about**
- Why does PySpark provide its own `sum()` and `count()` instead of relying on Python's built-ins?
- Which type classes will you need in Task 03 when defining an explicit CSV schema?
- What is the difference between `col("name")` and simply passing the string `"name"` to some DataFrame methods?

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, ____, DoubleType, ____, DateType
)
from pyspark.sql.functions import (
    col, lit, ____, input_file_name, when, year,
    count, sum, avg, ____
)

In [0]:
# ── CHECK 01 ──────────────────────────────────────────────────────────
assert 'StructType'         in dir(), "Missing: StructType"
assert 'col'                in dir(), "Missing: col"
assert 'current_timestamp'  in dir(), "Missing: current_timestamp"
assert 'when'               in dir(), "Missing: when"
assert 'desc'               in dir(), "Missing: desc"
print(" Task 01 passed — imports OK")

---
## Task 02 — Load CSV with automatic schema inference  Pre-filled

> **This cell is pre-filled — just run it.** It shows the standard pattern for loading a CSV with inferred schema. You will write the explicit-schema version yourself in Task 03.

Load `CUSTOMERS_CSV` using `inferSchema=true` and `header=true`.
Store the result in `customers_infer_df`.
Then print the schema and display the first 5 rows.

** Guidance — Task 02**

The goal is to load a CSV file into a Spark DataFrame by letting Spark determine each column's data type automatically.

**The DataFrameReader chain**
Every file read in PySpark follows the same builder pattern: `spark.read` returns a `DataFrameReader` you configure by chaining method calls. Two options are essential for CSV files: one that tells Spark the first row contains column names (rather than data values), and one that enables automatic schema inference. The final call takes the file path and returns the DataFrame.

**What inferSchema actually does**
When schema inference is enabled, Spark makes **two passes** over the file: one to sample the data and determine types, then another to actually read it. This doubles I/O cost. For small lab files this is acceptable, but in production you should always provide the schema explicitly — as you will do in Task 03.

**Verify the result**
After loading, call `printSchema()` to inspect the inferred types, then `display(...)` the first few rows to confirm the data looks correct.

**Things to think about**
- What data type do you expect Spark to infer for `customer_id`? What about `registration_date`?
- What would happen if the CSV contained a column with a mix of numeric and text values in different rows?

In [0]:
customers_infer_df = (
    spark.read
        .format("csv")
        .option("____", "true")
        .option("____", "true")
        .load(CUSTOMERS_CSV)
)

customers_infer_df.printSchema()
display(customers_infer_df.limit(5))

In [0]:
# ── CHECK 02 ──────────────────────────────────────────────────────────
assert 'customers_infer_df' in dir(), "customers_infer_df not defined"
assert customers_infer_df.count() > 0, "DataFrame is empty"
assert 'customer_id' in customers_infer_df.columns, "Missing column: customer_id"
assert 'email'       in customers_infer_df.columns, "Missing column: email"
print(f" Task 02 passed — {customers_infer_df.count():,} rows, {len(customers_infer_df.columns)} columns")

---
> ** Section 1 / 4 — Schema & File Loading · Tasks 03–05** &nbsp;|&nbsp; `░░░░░░░░░░` 2 / 12 done
>
> **Your first coding task starts here.** Tasks 01 & 02 above are pre-filled setup — make sure you ran them.
>
> After this section you will be able to: define an explicit `StructType`, load CSV and JSON files, and inspect any DataFrame.

---

---
## Task 03 — Load CSV with explicit schema

Define a `StructType` for the customer dataset and use it to load the CSV.
Store the schema in `customers_schema` and the DataFrame in `customers_df`.

Required columns and types:

| Column | Type | Nullable |
|--------|------|----------|
| customer_id | StringType | False |
| first_name | StringType | True |
| last_name | StringType | True |
| email | StringType | True |
| phone | StringType | True |
| city | StringType | True |
| state | StringType | True |
| country | StringType | True |
| registration_date | DateType | True |
| customer_segment | StringType | True |

** Guidance — Task 03**

The goal is to define a precise schema in code and use it to read the same CSV — eliminating the double-scan and guaranteeing predictable types.

**Building a StructType schema**
A `StructType` is a list of `StructField` objects. Each `StructField` takes three arguments: the column name (string), the data type (a type instance such as `StringType()`), and a boolean `nullable` flag. Notice that `customer_id` should have `nullable=False` because every customer must have an identifier.

**Which type for `registration_date`?**
The column holds dates in `YYYY-MM-DD` format. Use `DateType()` — Spark will automatically parse the standard ISO date format when reading CSV. If the format were non-standard, you would need to keep it as `StringType()` and convert afterwards.

**Using the schema on read**
Pass your `StructType` to `.schema(...)` in the reader chain **instead of** `.option("inferSchema", "true")`. The two approaches are mutually exclusive — use only one.

**Things to think about**
- What happens if the CSV contains a date value in a column you declared as `IntegerType()`?
- What is the benefit of setting `nullable=False` on `customer_id`?
- When would you prefer an explicit schema over `inferSchema` in a production pipeline?

In [0]:
# Step 1 — define customers_schema (StructType with 10 fields)
# Required columns: customer_id (not nullable), first_name, last_name, email, phone,
#                   city, state, country, registration_date (DateType), customer_segment

# YOUR CODE HERE
customers_schema = ...

# Step 2 — load CUSTOMERS_CSV using the explicit schema (no inferSchema)

# YOUR CODE HERE
customers_df = ...

customers_df.printSchema()
display(customers_df.limit(5))

In [0]:
# ── CHECK 03 ──────────────────────────────────────────────────────────
from pyspark.sql.types import DateType as _DT, StringType as _ST

assert 'customers_df'     in dir(), "customers_df not defined"
assert 'customers_schema' in dir(), "customers_schema not defined"
assert isinstance(customers_schema, StructType), "customers_schema must be StructType"

_schema_dict = {f.name: f.dataType for f in customers_df.schema.fields}
assert 'customer_id'        in _schema_dict, "Missing column: customer_id"
assert 'registration_date'  in _schema_dict, "Missing column: registration_date"
assert isinstance(_schema_dict['registration_date'], _DT), \
    f"registration_date must be DateType, got {_schema_dict['registration_date']}"


assert customers_df.count() > 0, "DataFrame is empty"
print(f" Task 03 passed — explicit schema OK, {customers_df.count():,} rows")

---
## Task 04 — Explore the customer DataFrame

Using `customers_df`, compute:
1. Total row count → store in `total_rows` (int)
2. Number of distinct countries → store in `distinct_countries` (int)
3. Summary statistics for all columns → store in `summary_df` (DataFrame from `.summary()`)

** Guidance — Task 04**

The goal is to compute three basic exploratory statistics about `customers_df`.

**Total row count**
The most fundamental DataFrame operation — call the method that returns the total number of rows as a plain Python integer. Assign it to `total_rows`.

**Distinct countries**
First project down to a single column using `.select("country")`, then chain `.distinct()` to keep only unique values, and finally `.count()` to get the number. The result is an integer.

**Statistical summary**
The `.summary()` method computes standard descriptive statistics for each column: count, mean, standard deviation, min, 25th/50th/75th percentiles, and max. It returns a DataFrame — assign it to `summary_df`. You can also try `.describe()` which returns a slightly smaller set of stats (no percentiles).

**Things to think about**
- What is the difference between `.summary()` and `.describe()`?
- Why might `distinct_countries` be lower than expected if the source data has inconsistent casing (`"USA"` vs `"usa"` vs `"Us"`)?

In [0]:
# Compute total_rows (int), distinct_countries (int), and summary_df (DataFrame)

# YOUR CODE HERE
total_rows = ...
distinct_countries = ...
summary_df = ...

print(f"Total rows          : {total_rows:,}")
print(f"Distinct countries  : {distinct_countries}")
display(summary_df)

In [0]:
# ── CHECK 04 ──────────────────────────────────────────────────────────
assert isinstance(total_rows, int) and total_rows > 0, \
    "total_rows must be a positive integer"
assert isinstance(distinct_countries, int) and distinct_countries == 1, \
    "distinct_countries must be > 1"
assert hasattr(summary_df, 'columns'), "summary_df must be a DataFrame"
print(f" Task 04 passed — {total_rows:,} rows, {distinct_countries} countries")

---
## Task 05 — Load JSON (Orders)

Load `ORDERS_JSON` using an explicit schema.
Store the schema in `orders_schema` and the DataFrame in `orders_df`.

Required columns:

| Column | Type |
|--------|------|
| order_id | StringType |
| customer_id | StringType |
| order_datetime | StringType | ← keep as String, cast in Silver |
| quantity | IntegerType |
| unit_price | DoubleType |
| total_amount | DoubleType |
| payment_method | StringType |
| status | StringType |

** Guidance — Task 05**

The goal is to define a schema and load the orders JSON file into a DataFrame.

**JSON Lines format**
`orders_batch.json` uses the JSON Lines format (also called NDJSON): each line is a self-contained, valid JSON object. Spark handles this natively — you just specify `format("json")` and supply the path. No special options are required for the basic single-file case.

**Building the schema**
Construct a `StructType` with a `StructField` for each column. Pay attention to numeric types: `quantity` uses `IntegerType()`, while `unit_price` and `total_amount` require `DoubleType()` because they hold decimal values. Notice that `order_datetime` must stay as `StringType()` for now — type-casting belongs in the Silver layer, not Bronze ingestion.

**Loading the file**
The reader chain follows the familiar pattern: `spark.read.format("json").schema(<your_schema>).load(ORDERS_JSON)`. Assign the result to `orders_df`.

**Things to think about**
- Why do we keep `order_datetime` as a string at the Bronze stage rather than parsing it to a timestamp immediately?
- What would Spark do if the JSON file contained a field that is not declared in your explicit schema?

In [0]:
# Step 1 — define orders_schema (StructType with 8 fields)
# Columns: order_id (String), customer_id (String), order_datetime (String — keep as raw),
#          quantity (Integer), unit_price (Double), total_amount (Double),
#          payment_method (String), status (String)

# YOUR CODE HERE
orders_schema = ...

# Step 2 — load ORDERS_JSON using the schema

# YOUR CODE HERE
orders_df = ...

orders_df.printSchema()
display(orders_df.limit(5))

In [0]:
# ── CHECK 05 ──────────────────────────────────────────────────────────
assert 'orders_df'     in dir(), "orders_df not defined"
assert 'orders_schema' in dir(), "orders_schema not defined"

_ocols = orders_df.columns
for _c in ['order_id', 'customer_id', 'total_amount', 'payment_method']:
    assert _c in _ocols, f"Missing column: {_c}"

_odict = {f.name: f.dataType for f in orders_df.schema.fields}
assert isinstance(_odict.get('total_amount'), DoubleType), \
    "total_amount must be DoubleType"
assert isinstance(_odict.get('order_datetime'), StringType), \
    "order_datetime must remain StringType in Bronze/this task"

assert orders_df.count() > 0, "orders_df is empty"
print(f" Task 05 passed — orders loaded: {orders_df.count():,} rows")

---
> ** Section 2 / 4 — DataFrame Transformations · Tasks 06–08** &nbsp;|&nbsp; `█████░░░░░` 5 / 12 done
>
> After this section you will be able to: filter rows with `filter` / `where`, add computed columns with `withColumn`, and write conditional logic with `when` / `otherwise`.

---

---
## Task 06 — Filter customers

From `customers_df` create two filtered DataFrames:

1. `customers_usa` — customers from the **USA** only
2. `customers_recent` — customers registered **in 2023 or later**
   (use the `year()` function on `registration_date`)

** Guidance — Task 06**

The goal is to create two filtered subsets of `customers_df` using two different kinds of conditions.

**filter() and where()**
`filter()` and `where()` are identical in PySpark — `where()` is simply an alias for developers who prefer SQL-style naming. Both accept a `Column` expression. Use `col("column_name")` to reference a column inside the expression.

**Filtering by equality**
For `customers_usa`, keep only rows where the `country` column equals the string `"USA"`. Use the `==` operator between the column reference and the literal string.

**Filtering with a function**
For `customers_recent`, you need customers registered in 2023 or later. The `year()` function extracts the integer year from a `DateType` column. Wrap `col("registration_date")` inside `year(...)`, then compare with `>= 2023`.

**Things to think about**
- What is the difference between `filter(col("country") == "USA")` and `filter("country = 'USA'")`?
- When combining two conditions with `&` or `|`, why must each condition be wrapped in parentheses?
- What happens to `customers_recent` if a row has a `null` in `registration_date`?

In [0]:
# Filter 1 — customers from USA only

# YOUR CODE HERE
customers_usa = ...

# Filter 2 — customers registered in 2024 or later (hint: use year() on registration_date)

# YOUR CODE HERE
customers_recent = ...

print(f"USA customers      : {customers_usa.count():,}")
print(f"Recent customers   : {customers_recent.count():,}")

In [0]:
# ── CHECK 06 ──────────────────────────────────────────────────────────
assert 'customers_usa'    in dir(), "customers_usa not defined"
assert 'customers_recent' in dir(), "customers_recent not defined"

_non_usa = customers_usa.filter(col("country") != "USA").count()
assert _non_usa == 0, f"customers_usa contains {_non_usa} non-USA rows"

_c_usa = customers_usa.count()
assert 0 < _c_usa == total_rows, "customers_usa must be a non-empty subset"

_old = customers_recent.filter(year(col("registration_date")) < 2023).count()
assert _old == 0, f"customers_recent contains {_old} rows from before 2023"

print(f" Task 06 passed — USA: {_c_usa:,} | recent: {customers_recent.count():,}")

---
## Task 07 — Add and transform columns

From `customers_df` create `customers_enriched` with two additional columns:

1. `full_name` — concatenation of `first_name + " " + last_name`
   (use string concatenation or `concat()` / `concat_ws()`)
2. `registration_year` — the year extracted from `registration_date` as IntegerType
   (use `year()` and `.cast(IntegerType())`)

** Guidance — Task 07**

The goal is to enrich `customers_df` with two computed columns, producing a new DataFrame called `customers_enriched`.

**DataFrames are immutable**
In PySpark, `.withColumn()` does **not** modify the existing DataFrame — it returns a new one with the additional column appended. You can chain multiple `.withColumn()` calls to add several columns in a single expression without creating intermediate variables.

**full_name — concatenating strings**
Use the `concat_ws(separator, col1, col2, ...)` function from `pyspark.sql.functions`. Passing `" "` (a single space) as the separator between `first_name` and `last_name` produces the expected `"Alice Smith"` format. `concat_ws` gracefully handles `null` values by ignoring them.

**registration_year — extracting year and casting**
The `year()` function extracts the year component from a `DateType` column and returns it as an integer. Wrap the result in `.cast(IntegerType())` to make the type explicit in the schema — this is good practice even when Spark might already infer `IntegerType`.

**Things to think about**
- What would `concat_ws(" ", col("first_name"), col("last_name"))` produce for a row where `first_name` is `null`?
- Why is it safer to chain `.withColumn()` calls rather than reassigning the same variable name at each step?

In [0]:
from pyspark.sql.functions import concat_ws

# Build customers_enriched with:
#   full_name         = concat_ws(" ", first_name, last_name)
#   registration_year = year(registration_date).cast(IntegerType())

# YOUR CODE HERE
customers_enriched = ...

display(customers_enriched.select("customer_id", "full_name", "registration_year").limit(5))

In [0]:
# ── CHECK 07 ──────────────────────────────────────────────────────────
from pyspark.sql.types import IntegerType as _IntT

assert 'customers_enriched' in dir(), "customers_enriched not defined"
_ecols = customers_enriched.columns
assert 'full_name'          in _ecols, "Missing column: full_name"
assert 'registration_year'  in _ecols, "Missing column: registration_year"

_edict = {f.name: f.dataType for f in customers_enriched.schema.fields}
assert isinstance(_edict['registration_year'], _IntT), \
    f"registration_year must be IntegerType, got {_edict['registration_year']}"

# .collect() is OK here: limited to 50 rows for assertion check
_sample = customers_enriched.select("first_name", "last_name", "full_name").limit(50).collect()
for row in _sample:
    if row.first_name and row.last_name:
        expected = f"{row.first_name} {row.last_name}"
        assert row.full_name == expected, \
            f"full_name mismatch: expected '{expected}', got '{row.full_name}'"

print(" Task 07 passed — full_name and registration_year OK")

---
## Task 08 — Conditional column: customer tier

Add a `customer_tier` column to `customers_enriched`.
Store the result in `customers_tiered`.

Rules:

| Condition | Tier |
|-----------|------|
| `registration_year` >= 2023 | `"New"` |
| `registration_year` >= 2021 | `"Regular"` |
| otherwise | `"Loyal"` |

** Guidance — Task 08**

The goal is to add a categorical column whose value depends on a series of conditions evaluated in order.

**The when().when().otherwise() chain**
PySpark's `when(condition, value)` is called as a **standalone function** (not a method on a column), then additional conditions are chained with `.when(...)`, and finally `.otherwise(value)` provides the default case. The **first matching branch wins** — evaluation stops at the first true condition.

**Condition order matters**
Notice that the `>= 2023` check comes before `>= 2021`. If the order were reversed, customers registered in 2023 or 2024 would incorrectly match the first `>= 2021` branch and receive `"Regular"` instead of `"New"`.

**Always include .otherwise()**
Without `.otherwise()`, rows that do not match any `when()` branch receive `null`. The CHECK cell validates there are no null values in `customer_tier`, so you must include the fallback.

**Things to think about**
- What tier would a customer with `registration_year = null` receive — with and without `.otherwise()`?
- Can you express the same logic using a SQL `CASE WHEN ... THEN ... ELSE ... END` expression inside `spark.sql()`?

In [0]:
# Add customer_tier column using when().when().otherwise() on registration_year:
#   >= 2024 → "New"  |  >= 2021 → "Regular"  |  otherwise → "Loyal"

# YOUR CODE HERE
customers_tiered = ...

display(customers_tiered.groupBy("customer_tier").count().orderBy("customer_tier"))

In [0]:
# ── CHECK 08 ──────────────────────────────────────────────────────────
assert 'customers_tiered' in dir(), "customers_tiered not defined"
assert 'customer_tier'    in customers_tiered.columns, \
    "Missing column: customer_tier"

_tiers = {r['customer_tier'] for r in customers_tiered.select("customer_tier").distinct().collect()}
assert _tiers <= {"New", "Regular", "Loyal"}, \
    f"Unexpected tier values: {_tiers}"
assert len(_tiers) >= 2, "Expected at least 2 distinct tiers in the dataset"

_null_tiers = customers_tiered.filter(col("customer_tier").isNull()).count()
assert _null_tiers == 0, f"customer_tier has {_null_tiers} null values — add .otherwise()"

print(f" Task 08 passed — tiers: {sorted(_tiers)}")

---
> ** Section 3 / 4 — Aggregations & SQL · Tasks 09–10** &nbsp;|&nbsp; `████████░░` 8 / 12 done
>
> After this section you will be able to: group and aggregate data with `groupBy().agg()`, register temp views, and query them with `spark.sql`.

---

---
## Task 09 — Aggregations

Using `orders_df`, compute the following aggregations in a **single** `groupBy().agg()` call.
Group by `payment_method`. Store the result in `payment_stats`.

Required columns in the result:

| Column | Aggregation |
|--------|-------------|
| `payment_method` | group key |
| `order_count` | COUNT(*) |
| `total_revenue` | SUM(total_amount) |
| `avg_order_value` | AVG(total_amount) |

Order the result by `total_revenue` descending.

** Guidance — Task 09**

The goal is to compute three different aggregations over `orders_df` grouped by `payment_method` — all in a single `groupBy().agg()` call.

**The groupBy().agg() pattern**
`groupBy("column")` partitions the DataFrame into groups based on unique values of that column. The subsequent `.agg(...)` call specifies one or more aggregation functions to apply to each group. You can pass multiple aggregation expressions as separate arguments to a single `.agg(...)` call.

**Naming the result columns**
Each aggregation expression must be given a meaningful name using `.alias(...)`. For example: `count("*").alias("order_count")`. Without an alias, Spark generates default names like `count(1)` which are not user-friendly and will cause the CHECK to fail.

**Three aggregation functions**
You need: one that counts all rows in each group, one that sums a numeric column, and one that computes the average. All three are already imported from `pyspark.sql.functions` via Task 01.

**Ordering the result**
Chain `.orderBy(desc("total_revenue"))` after `.agg(...)` to sort from highest to lowest revenue.

**Things to think about**
- What is the difference between `count("*")` and `count("total_amount")`?
- What would the result look like if you called `sum("total_amount")` without an alias?

In [0]:
# groupBy payment_method, compute order_count / total_revenue / avg_order_value
# Order by total_revenue DESC

# YOUR CODE HERE
payment_stats = ...

display(payment_stats)

In [0]:
# ── CHECK 09 ──────────────────────────────────────────────────────────
assert 'payment_stats' in dir(), "payment_stats not defined"

_pcols = payment_stats.columns
for _c in ['payment_method', 'order_count', 'total_revenue', 'avg_order_value']:
    assert _c in _pcols, f"Missing column in payment_stats: {_c}"

_all_methods = {r['payment_method'] for r in orders_df.select("payment_method").distinct().collect()}
_stat_methods = {r['payment_method'] for r in payment_stats.select("payment_method").collect()}
assert _all_methods == _stat_methods, "payment_stats does not cover all payment methods"

_totals = [r['total_revenue'] for r in payment_stats.collect()]
assert _totals == sorted(_totals, reverse=True), "Result must be ordered by total_revenue DESC"

print(f" Task 09 passed — {len(_stat_methods)} payment methods aggregated")

---
## Task 10 — Temporary views and SQL

1. Register `customers_df` as a temp view named `"customers_view"`
2. Register `orders_df` as a temp view named `"orders_view"`
3. Using `spark.sql`, write a query that returns the **top 5 countries** by number of customers,
   with columns `country` and `customer_count`. Store the result in `top_countries`.

** Guidance — Task 10**

The goal is to expose DataFrames as SQL-queryable tables within the session, then run a SQL aggregation query against them.

**createOrReplaceTempView**
Calling `df.createOrReplaceTempView("name")` registers the DataFrame as a named logical table visible only within the current Spark session. No data is copied or written to disk — it is simply a named reference. If a view with the same name already exists, it is silently replaced.

**Querying with spark.sql()**
Once a temp view is registered, reference it by name inside any `spark.sql("SELECT ...")` call. The result is a regular DataFrame. Write a standard `SELECT … GROUP BY … ORDER BY … LIMIT` query to find the top 5 countries by customer count.

**What to name your columns**
The CHECK cell expects a column named `customer_count` in `top_countries`. Use `COUNT(*) AS customer_count` in your query to ensure the right name.

**Things to think about**
- Why does a temp view disappear when the Spark cluster is restarted?
- What is the difference between a temp view created with `createOrReplaceTempView` and a permanent view created with `CREATE VIEW ... AS SELECT ...` stored in Unity Catalog?
- Can you join `customers_view` and `orders_view` in the same SQL query? What SQL clause would you use?

In [0]:
# Step 1 — register both DataFrames as temp views

# YOUR CODE HERE

# Step 2 — query top 5 countries by customer count (column must be named 'customer_count')

# YOUR CODE HERE
top_countries = ...

display(top_countries)

In [0]:
# ── CHECK 10 ─────────────────────────────────────────────────────────────────
assert 'top_countries' in dir(), "top_countries not defined"

_views = {r.viewName for r in spark.sql("SHOW VIEWS").collect()}
assert 'customers_view' in _views, "Temp view 'customers_view' not found"
assert 'orders_view'    in _views, "Temp view 'orders_view' not found"

assert 'country'         in top_countries.columns, "Missing column: country"
assert 'customer_count'  in top_countries.columns, "Missing column: customer_count"
assert top_countries.count() == 5, f"Expected 5 rows, got {top_countries.count()}"

_counts = [r['customer_count'] for r in top_countries.collect()]
assert _counts == sorted(_counts, reverse=True), "Result must be ordered by customer_count DESC"

print(f" Task 10 passed — views registered, top countries: {[r['country'] for r in top_countries.collect()]}")

---
> ** Section 4 / 4 — Write to Delta + Bonus · Tasks 11–12** &nbsp;|&nbsp; `██████████` 10 / 12 done
>
> **Almost there!** Task 11 finalises the Bronze pipeline — you'll add metadata columns and persist the table. Task 12 ⭐ is a bonus join: attempt it if you have time left.

---

---
## Task 11 — Write Bronze Delta table

Write `customers_tiered` to a managed Delta table at `TARGET_TABLE`.

Requirements:
- Mode: `overwrite`
- Format: `delta`
- Allow schema changes on re-run: `overwriteSchema = true`
- Add metadata columns first:
  - `_load_ts` — current load timestamp (`current_timestamp()`)
  - `_source_file` — source file path from metadata (`col("_metadata.file_path")`)

Store the final DataFrame (before write) in `customers_bronze`.

** Guidance — Task 11**

The goal is to finalise the Bronze layer DataFrame and persist it as a managed Delta table.

**Bronze layer rule**
The Bronze layer stores **raw, unmodified data plus only metadata**. You must not apply any business logic, filtering, or type casting at this stage. Two metadata columns are added to every Bronze table: `_load_ts` (the exact timestamp of the load) and `_source_file` (the source file path for lineage tracking).

**Adding metadata columns**
Use `.withColumn(...)` to attach both columns. `current_timestamp()` returns the current system time as a `TimestampType` constant column. 

For `_source_file`, use `col("_metadata.file_path")` to extract the source file path from Spark's built-in metadata. Every DataFrame loaded from files automatically includes a hidden `_metadata` struct column with fields like `file_path`, `file_name`, `file_size`, and `file_modification_time`. Referencing `col("_metadata.file_path")` promotes this hidden column into your result, capturing the actual source location dynamically.

**Writing to Delta**
Chain onto the final DataFrame: `.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE)`. The `saveAsTable()` method registers the table in Unity Catalog as a **managed** table — Databricks controls both the metadata and the physical data files. The `overwriteSchema` option allows the table definition to change across re-runs of this cell.

**Things to think about**
- What is the difference between `.saveAsTable(TABLE)` and `.save("/path/to/dir")`?
- Why do Bronze metadata column names conventionally start with `_`?
- What other fields are available in the `_metadata` struct column?

In [0]:
# Step 1 — add metadata columns: _load_ts (current_timestamp) and _source_file (col("_metadata.file_path"))

# YOUR CODE HERE
customers_bronze = ...

# Step 2 — write customers_bronze as a managed Delta table at TARGET_TABLE
# (overwrite mode, overwriteSchema=true)

# YOUR CODE HERE

print(f"Written to: {TARGET_TABLE}")
display(spark.table(TARGET_TABLE).limit(5))

In [0]:
# ── CHECK 11 ─────────────────────────────────────────────────────────────────
from pyspark.sql.types import TimestampType as _TsT

assert 'customers_bronze' in dir(), "customers_bronze not defined"

_tbl = spark.table(TARGET_TABLE)
_tcols = _tbl.columns
assert '_load_ts'     in _tcols, "Missing column: _load_ts"
assert '_source_file' in _tcols, "Missing column: _source_file"
assert 'customer_tier' in _tcols, "Missing column: customer_tier (from Task 08)"

_tdict = {f.name: f.dataType for f in _tbl.schema.fields}
assert isinstance(_tdict['_load_ts'], _TsT), "_load_ts must be TimestampType"

_row_count = _tbl.count()
assert _row_count > 0, "Delta table is empty"

_src_vals = {r['_source_file'] for r in _tbl.select("_source_file").distinct().collect()}
assert any('customers.csv' in str(path) for path in _src_vals if path), \
    "_source_file paths must contain 'customers.csv'"

print(f" Task 11 passed — {_row_count:,} rows written to {TARGET_TABLE}")

---
## Task 12 ⭐ BONUS — Join orders with customers

Create `orders_enriched` by joining `orders_df` (left) with `customers_df` (right)
on `customer_id`. Use a **left join** so that orders without a matching customer are kept.

Select only these columns in the result:
`order_id`, `customer_id`, `first_name`, `last_name`, `country`, `total_amount`, `payment_method`

Then compute `orders_by_country`: count of orders and total revenue per `country`,
ordered by `total_revenue` DESC.

** Guidance — Task 12**

The goal is to enrich the orders data with customer information, then aggregate the result by country.

**Left join semantics**
A left join keeps **all rows** from the left (primary) DataFrame and attaches the matching row from the right (reference) DataFrame. Orders with no matching `customer_id` in `customers_df` are still included — their customer columns will be `null`. This is the right choice for analytics: you should never silently lose orders.

**Handling ambiguous column names**
Both DataFrames have a `customer_id` column. After the join, the result technically contains two columns with that name. To avoid an `AnalysisException: ambiguous reference`, use the DataFrame-qualified notation `orders_df["customer_id"]` and `customers_df["first_name"]` etc. — rather than the bare `col("customer_id")` — in both the join condition and the subsequent `.select(...)` call.

**Selecting the needed columns**
Chain `.select(...)` immediately after `.join(...)` to keep only the seven required columns. Qualify each reference with the source DataFrame to resolve any ambiguity.

**The country aggregation**
Once `orders_enriched` is built, use `groupBy("country").agg(...)` to compute order count and total revenue per country, ordered by revenue descending. Name both aggregated columns explicitly with `.alias(...)`.

**Things to think about**
- What join type would you use if you only wanted orders where the customer is known (i.e., drop unmatched orders)?
- The CHECK cell asserts that `orders_enriched.count() == orders_df.count()`. Why doesn't this hold for a left join?

In [0]:
# Step 1 — left join orders_df with customers_df on customer_id
# Use df["col"] notation (not col("col")) to resolve the ambiguous customer_id column
# Select: order_id, customer_id, first_name, last_name, country, total_amount, payment_method

# YOUR CODE HERE
orders_enriched = ...

display(orders_enriched)

In [0]:
# Step 2 — group by country: compute order_count and total_revenue (DESC)

# YOUR CODE HERE
orders_by_country = ...

display(orders_by_country.limit(10))

In [0]:
# ── CHECK 12 ──────────────────────────────────────────────────────────
assert 'orders_enriched'   in dir(), "orders_enriched not defined"
assert 'orders_by_country' in dir(), "orders_by_country not defined"

_ecols = orders_enriched.columns
for _c in ['order_id', 'customer_id', 'first_name', 'country', 'total_amount']:
    assert _c in _ecols, f"Missing column in orders_enriched: {_c}"

assert orders_enriched.count() > orders_df.count(), \
    "Left join must preserve all orders and more due to dupliate value"

assert 'total_revenue' in orders_by_country.columns or \
       'total_amount'  in orders_by_country.columns, \
    "orders_by_country must have a revenue column"

print(f" Task 12 BONUS passed — {orders_enriched.count():,} enriched orders, "
      f"{orders_by_country.count()} countries")

---
##  Workshop Complete!

**Next:** `04_delta_fundamentals.ipynb` — Schema Enforcement, CRUD, Time Travel, RESTORE
